### **Semana 5 - BM25, recuperación híbrida, reranking y RAG**

#### **Pregunta experimental**

> ¿Cómo cambia la recuperación de evidencia cuando se compara BM25, recuperación densa, recuperación híbrida mediante RRF y recuperación híbrida con reranking, manteniendo constantes corpus, consultas, qrels, segmentación y top-k?.

La continuidad es:

```text
Semana 4
fragmentos -> E5 -> FAISS -> dense ranking
                          |
                          v
Semana 5
BM25 --------------------+
                          +-> RRF -> Cross-Encoder -> top-k -> contexto -> LLM
Dense de Semana 4 -------+
```

No se modifica el benchmark después de observar los resultados.

#### **0. Protocolo antes de ejecutar**

Completa antes de observar las métricas:

```text
Pregunta:

Hipótesis:

Baseline conceptual:
dense retrieval de Semana 4

Condiciones:
A = BM25
B = dense
C = hybrid RRF
D = hybrid RRF + reranker

Variables fijas:
corpus
queries
qrels
target_words = 180
overlap_passages = 1
dense model
top-k
candidate_depth
RRF constant
reranker model

Métrica principal:
mean Recall@3

Evidencia secundaria:
latencia media y p95
IC 95% por bootstrap pareado sobre Recall@3
```

La comparación principal de Semana 5 sigue siendo de **retrieval**.

El bloque RAG final añade tres controles didácticos:

```text
closed-book
RAG con contexto recuperado
control negativo sin evidencia esperada
```

También audita sintácticamente las citas emitidas por el LLM. Esta auditoría comprueba si un `chunk_id` citado pertenece al contexto entregado; **no demuestra por sí sola groundedness semántico**. La evaluación exhaustiva de grounding permanece en Semana 7.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

SEED = 42
np.random.seed(SEED)

DENSE_MODEL_ID = "intfloat/multilingual-e5-small"
RERANKER_MODEL_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
GENERATOR_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

DENSE_MODEL_REVISION = os.environ.get("CC0F4_DENSE_MODEL_REVISION")
RERANKER_MODEL_REVISION = os.environ.get("CC0F4_RERANKER_MODEL_REVISION")
GENERATOR_MODEL_REVISION = os.environ.get("CC0F4_GENERATOR_MODEL_REVISION")

RUN_REAL_RETRIEVAL = os.environ.get("CC0F4_RUN_REAL_RETRIEVAL", "1") == "1"
RUN_REAL_RERANKER = os.environ.get("CC0F4_RUN_REAL_RERANKER", "1") == "1"
RUN_REAL_LLM = os.environ.get("CC0F4_RUN_REAL_LLM", "1") == "1"

TARGET_WORDS = 180
OVERLAP_PASSAGES = 1
CANDIDATE_DEPTH = 10
TOP_K_VALUES = [1, 3, 5]
RRF_CONSTANT = 60
RAG_TOP_K = 3
RAG_DEMO_QUERY_IDS = ["Q04", "Q21"]
BOOTSTRAP_RESAMPLES = 5000

NEGATIVE_CONTROL = {
    "query_id": "NEG-01",
    "query": "¿Cuál es el precio actual del menú de la cafetería del campus?",
    "expected": "EVIDENCIA_INSUFICIENTE",
}

print("DENSE_MODEL_ID:", DENSE_MODEL_ID)
print("RERANKER_MODEL_ID:", RERANKER_MODEL_ID)
print("GENERATOR_MODEL_ID:", GENERATOR_MODEL_ID)
print("RUN_REAL_RETRIEVAL:", RUN_REAL_RETRIEVAL)
print("RUN_REAL_RERANKER:", RUN_REAL_RERANKER)
print("RUN_REAL_LLM:", RUN_REAL_LLM)


#### **1. Reutilizar el benchmark de Semana 4**

Semana 5 no crea un dataset nuevo. Reutiliza:

```text
Semana4/datos/corpus_semana4.jsonl
Semana4/datos/queries_semana4.jsonl
Semana4/datos/qrels_semana4.json
```

Los qrels continúan definidos sobre `passage_id`, de modo que la evidencia relevante no depende del `chunk_id`.

In [ ]:
def resolve_data_dir() -> Path:
    explicit = os.environ.get("CC0F4_SEMANA5_DATA")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if path.is_dir():
            return path
        raise FileNotFoundError(f"No existe CC0F4_SEMANA5_DATA={path}")

    cwd = Path.cwd().resolve()
    candidates = []
    for root in [cwd, *cwd.parents]:
        candidates.extend([
            root / "Semana4" / "datos",
            root / "datos",
        ])

    for candidate in candidates:
        if (
            (candidate / "corpus_semana4.jsonl").is_file()
            and (candidate / "queries_semana4.jsonl").is_file()
            and (candidate / "qrels_semana4.json").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontraron los datos de Semana 4. "
        "Ejecuta desde CC-0F4 o define CC0F4_SEMANA5_DATA."
    )


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido en {path}, línea {line_number}"
                ) from exc
    return rows


DATA_DIR = resolve_data_dir()
documents = read_jsonl(DATA_DIR / "corpus_semana4.jsonl")
queries = read_jsonl(DATA_DIR / "queries_semana4.jsonl")
qrels = json.loads(
    (DATA_DIR / "qrels_semana4.json").read_text(encoding="utf-8")
)

print("DATA_DIR:", DATA_DIR)
print("Documentos:", len(documents))
print("Consultas:", len(queries))
print("Qrels:", len(qrels))

In [ ]:
def validate_dataset(
    documents: list[dict[str, Any]],
    queries: list[dict[str, Any]],
    qrels: dict[str, list[str]],
) -> None:
    doc_ids = [doc["doc_id"] for doc in documents]
    query_ids = [row["query_id"] for row in queries]

    if len(doc_ids) != len(set(doc_ids)):
        raise ValueError("Hay doc_id duplicados.")
    if len(query_ids) != len(set(query_ids)):
        raise ValueError("Hay query_id duplicados.")
    if set(query_ids) != set(qrels):
        raise ValueError("Cada query debe tener una entrada en qrels.")

    empty_qrels = sorted(qid for qid, relevant in qrels.items() if not relevant)
    if empty_qrels:
        raise ValueError(
            "Qrels vacíos. Recall@k no está definido para estas consultas: "
            f"{empty_qrels}"
        )

    passage_ids = set()
    for doc in documents:
        passages = doc.get("passages") or []
        if not passages:
            raise ValueError(f"{doc['doc_id']} no contiene passages.")
        for passage in passages:
            pid = passage["passage_id"]
            if pid in passage_ids:
                raise ValueError(f"passage_id duplicado: {pid}")
            passage_ids.add(pid)

    missing = {
        pid
        for relevant in qrels.values()
        for pid in relevant
        if pid not in passage_ids
    }
    if missing:
        raise ValueError(f"Qrels referencia passages inexistentes: {sorted(missing)}")


validate_dataset(documents, queries, qrels)
print("Integridad del benchmark: OK")


#### **2. Segmentación fija: baseline de Semana 4**

La variable experimental ya no es el tamaño del fragmento.

```text
target_words = 180
overlap_passages = 1
```

Todos los recuperadores reciben exactamente los mismos fragmentos.

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())


def build_chunks(
    documents: list[dict[str, Any]],
    target_words: int,
    overlap_passages: int,
) -> list[dict[str, Any]]:
    chunks = []

    for doc in documents:
        passages = doc["passages"]
        start = 0
        chunk_number = 1

        while start < len(passages):
            selected = []
            total_words = 0
            end = start

            while end < len(passages):
                passage = passages[end]
                n_words = word_count(passage["text"])

                if selected and total_words + n_words > target_words:
                    break

                selected.append(passage)
                total_words += n_words
                end += 1

            if not selected:
                selected = [passages[start]]
                end = start + 1
                total_words = word_count(selected[0]["text"])

            chunks.append({
                "chunk_id": f"{doc['doc_id']}-C{chunk_number:03d}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "passage_ids": [p["passage_id"] for p in selected],
                "n_words": total_words,
                "text": " ".join(p["text"] for p in selected),
            })

            if end >= len(passages):
                break

            next_start = max(start + 1, end - overlap_passages)
            start = next_start
            chunk_number += 1

    return chunks


chunks = build_chunks(
    documents,
    target_words=TARGET_WORDS,
    overlap_passages=OVERLAP_PASSAGES,
)

print("Fragmentos:", len(chunks))
print("Palabras promedio:", round(np.mean([c["n_words"] for c in chunks]), 2))
print("Máximo de palabras:", max(c["n_words"] for c in chunks))

#### **3. Recuperación dispersa con BM25**

BM25 utiliza coincidencia léxica con saturación de frecuencia y normalización por longitud.

La ejecución canónica usa `rank-bm25`. El fallback local existe únicamente para validación offline. Si se solicita una corrida canónica y `rank-bm25` no está instalado, el cuaderno falla con un mensaje explícito en lugar de sustituir silenciosamente el componente.

Los empates se resuelven por `chunk_id` para que el ranking sea determinista bajo scores idénticos.


In [ ]:
def lexical_tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", text.lower(), flags=re.UNICODE)


def stable_desc_indices(
    scores: np.ndarray,
    items: list[dict[str, Any]],
    depth: int,
) -> list[int]:
    if depth <= 0:
        return []
    return sorted(
        range(len(scores)),
        key=lambda idx: (-float(scores[idx]), items[idx]["chunk_id"]),
    )[: min(depth, len(scores))]


class LocalBM25:
    """Fallback de validación. La ejecución canónica usa rank-bm25."""

    def __init__(self, corpus_tokens: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.corpus_tokens = corpus_tokens
        self.k1 = k1
        self.b = b
        self.doc_len = np.array([len(doc) for doc in corpus_tokens], dtype=float)
        self.avgdl = float(self.doc_len.mean()) if len(self.doc_len) else 0.0
        self.doc_freq = Counter()
        for doc in corpus_tokens:
            self.doc_freq.update(set(doc))
        self.n_docs = len(corpus_tokens)

    def get_scores(self, query_tokens: list[str]) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=float)
        for i, doc in enumerate(self.corpus_tokens):
            tf = Counter(doc)
            for term in query_tokens:
                df = self.doc_freq.get(term, 0)
                if df == 0:
                    continue
                idf = math.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5))
                freq = tf.get(term, 0)
                denom = freq + self.k1 * (
                    1.0 - self.b + self.b * self.doc_len[i] / max(self.avgdl, 1e-12)
                )
                if denom > 0:
                    scores[i] += idf * freq * (self.k1 + 1.0) / denom
        return scores


try:
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([lexical_tokenize(c["text"]) for c in chunks])
    BM25_IMPLEMENTATION = "rank-bm25"
except ImportError as exc:
    if RUN_REAL_RETRIEVAL:
        raise ImportError(
            "Falta 'rank-bm25' para la ejecución canónica. "
            "Instala el entorno global del curso o exporta "
            "CC0F4_RUN_REAL_RETRIEVAL=0 para validar offline."
        ) from exc
    bm25 = LocalBM25([lexical_tokenize(c["text"]) for c in chunks])
    BM25_IMPLEMENTATION = "fallback-local-validation"

print("BM25_IMPLEMENTATION:", BM25_IMPLEMENTATION)


def bm25_ranking(query: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    scores = np.asarray(bm25.get_scores(lexical_tokenize(query)), dtype=float)
    order = stable_desc_indices(scores, chunks, depth)
    return [
        {
            **chunks[int(idx)],
            "score": float(scores[int(idx)]),
            "rank": rank,
            "source": "bm25",
        }
        for rank, idx in enumerate(order, start=1)
    ]


#### **4. Recuperación densa heredada de Semana 4**

Se mantiene:

```text
intfloat/multilingual-e5-small
query:
passage:
normalización L2
FAISS IndexFlatIP
```

El modo offline utiliza TF-IDF + SVD exclusivamente como sustituto de validación.

En modo canónico, la ausencia de `sentence-transformers` o `faiss` produce un mensaje pedagógico explícito. No se sustituye silenciosamente el recuperador real.

Para medir latencia de consulta, el embedding de cada query se calcula dentro de `dense_ranking`; la construcción del índice y la carga del modelo quedan fuera de la medición.


In [ ]:
class OfflineDenseEncoder:
    def __init__(self, seed: int = 42):
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            lowercase=True,
        )
        self.seed = seed
        self.svd = None

    def fit_documents(self, texts: list[str]) -> np.ndarray:
        sparse = self.vectorizer.fit_transform(texts)
        n_components = min(
            64,
            sparse.shape[0] - 1,
            sparse.shape[1] - 1,
        )
        if n_components < 2:
            raise ValueError("Corpus demasiado pequeño para SVD de validación.")
        self.svd = TruncatedSVD(
            n_components=n_components,
            random_state=self.seed,
        )
        dense = self.svd.fit_transform(sparse)
        return normalize(dense, norm="l2").astype("float32")

    def encode_queries(self, texts: list[str]) -> np.ndarray:
        if self.svd is None:
            raise RuntimeError("El encoder offline no está ajustado.")
        sparse = self.vectorizer.transform(texts)
        dense = self.svd.transform(sparse)
        return normalize(dense, norm="l2").astype("float32")


chunk_texts = [c["text"] for c in chunks]

if RUN_REAL_RETRIEVAL:
    try:
        from sentence_transformers import SentenceTransformer
        import faiss
    except ImportError as exc:
        raise ImportError(
            "La recuperación densa canónica requiere 'sentence-transformers' "
            "y 'faiss'. Instala el entorno global del curso o exporta "
            "CC0F4_RUN_REAL_RETRIEVAL=0 para validar offline."
        ) from exc

    dense_kwargs = {}
    if DENSE_MODEL_REVISION:
        dense_kwargs["revision"] = DENSE_MODEL_REVISION

    dense_encoder = SentenceTransformer(DENSE_MODEL_ID, **dense_kwargs)

    document_inputs = [f"passage: {text}" for text in chunk_texts]

    max_doc_tokens = max(
        len(dense_encoder.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
        for text in document_inputs
    )
    if max_doc_tokens > dense_encoder.max_seq_length:
        raise ValueError(
            f"Un fragmento excede el límite del encoder: {max_doc_tokens} > "
            f"{dense_encoder.max_seq_length}."
        )

    document_vectors = dense_encoder.encode(
        document_inputs,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype("float32")

    dense_index = faiss.IndexFlatIP(document_vectors.shape[1])
    dense_index.add(document_vectors)
    DENSE_IMPLEMENTATION = "E5 + FAISS IndexFlatIP"
else:
    dense_encoder = OfflineDenseEncoder(SEED)
    document_vectors = dense_encoder.fit_documents(chunk_texts)
    dense_index = None
    DENSE_IMPLEMENTATION = "TF-IDF + SVD, solo validación"

print("DENSE_IMPLEMENTATION:", DENSE_IMPLEMENTATION)
print("Dimensión:", document_vectors.shape[1])


In [ ]:
query_by_id = {
    q["query_id"]: q
    for q in queries
}


def encode_dense_query(query_id: str) -> np.ndarray:
    query_text = query_by_id[query_id]["query"]

    if RUN_REAL_RETRIEVAL:
        vector = dense_encoder.encode(
            [f"query: {query_text}"],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return np.asarray(vector, dtype="float32")

    return dense_encoder.encode_queries([query_text])


def dense_ranking(query_id: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth <= 0:
        return []

    qvec = encode_dense_query(query_id)

    if RUN_REAL_RETRIEVAL:
        # IndexFlatIP es exacto. Para este benchmark pequeño se solicita el
        # ranking completo y después se aplica el corte con desempate estable.
        search_k = int(dense_index.ntotal)
        scores, indices = dense_index.search(qvec.astype("float32"), search_k)

        pairs = [
            (int(idx), float(score))
            for idx, score in zip(indices[0], scores[0])
            if int(idx) >= 0
        ]
        pairs.sort(
            key=lambda pair: (-pair[1], chunks[pair[0]]["chunk_id"])
        )
        pairs = pairs[: min(depth, len(pairs))]
    else:
        scores_all = document_vectors @ qvec[0]
        order = stable_desc_indices(scores_all, chunks, depth)
        pairs = [(int(idx), float(scores_all[int(idx)])) for idx in order]

    # Defensa adicional: FAISS documenta -1 como relleno cuando faltan vecinos.
    if any(idx < 0 for idx, _ in pairs):
        raise RuntimeError("FAISS devolvió un índice -1 no filtrado.")

    return [
        {
            **chunks[idx],
            "score": score,
            "rank": rank,
            "source": "dense",
        }
        for rank, (idx, score) in enumerate(pairs, start=1)
    ]


#### **5. Fusión híbrida con RRF**

BM25 y dense retrieval no producen scores naturalmente comparables.

RRF fusiona posiciones:

$$
RRF(d)=\sum_r \frac{1}{k+r(d)}.
$$

Se fija:

```text
k = 60
candidate_depth = 10
```

No se reajustan después de observar las métricas.

In [ ]:
def reciprocal_rank_fusion(
    rankings: list[list[dict[str, Any]]],
    depth: int = CANDIDATE_DEPTH,
    rank_constant: int = RRF_CONSTANT,
) -> list[dict[str, Any]]:
    scores = defaultdict(float)
    payload = {}
    sources = defaultdict(list)

    for ranking in rankings:
        for item in ranking:
            cid = item["chunk_id"]
            scores[cid] += 1.0 / (rank_constant + item["rank"])
            payload[cid] = {
                key: value
                for key, value in item.items()
                if key not in {"score", "rank", "source"}
            }
            sources[cid].append(item["source"])

    ordered = sorted(
        scores,
        key=lambda cid: (-scores[cid], cid),
    )[:depth]

    return [
        {
            **payload[cid],
            "score": float(scores[cid]),
            "rank": rank,
            "source": "+".join(sorted(set(sources[cid]))),
        }
        for rank, cid in enumerate(ordered, start=1)
    ]

#### **6. Reranking con Cross-Encoder**

El Cross-Encoder procesa conjuntamente:

```text
(query, candidate)
```

y produce un score de relevancia.

Se utiliza únicamente sobre los candidatos de RRF. El modo offline usa un sustituto léxico determinista para validar el flujo.

En modo canónico, si falta `sentence-transformers`, el cuaderno falla con una instrucción explícita. Los empates del reranker se resuelven por `chunk_id`.


In [ ]:
def offline_rerank_score(query: str, text: str) -> float:
    q = set(lexical_tokenize(query))
    d = set(lexical_tokenize(text))
    if not q:
        return 0.0
    return len(q & d) / len(q)


if RUN_REAL_RERANKER:
    try:
        from sentence_transformers import CrossEncoder
    except ImportError as exc:
        raise ImportError(
            "El reranker canónico requiere 'sentence-transformers'. "
            "Instala el entorno global del curso o exporta "
            "CC0F4_RUN_REAL_RERANKER=0 para validar offline."
        ) from exc

    reranker_kwargs = {"max_length": 512}
    if RERANKER_MODEL_REVISION:
        reranker_kwargs["revision"] = RERANKER_MODEL_REVISION

    reranker_model = CrossEncoder(RERANKER_MODEL_ID, **reranker_kwargs)
    RERANKER_IMPLEMENTATION = RERANKER_MODEL_ID
else:
    reranker_model = None
    RERANKER_IMPLEMENTATION = "lexical-overlap, solo validación"

print("RERANKER_IMPLEMENTATION:", RERANKER_IMPLEMENTATION)


def rerank_candidates(
    query: str,
    candidates: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    if not candidates:
        return []

    if RUN_REAL_RERANKER:
        pairs = [(query, item["text"]) for item in candidates]

        reranker_limit = int(getattr(reranker_model, "max_seq_length", 512) or 512)
        max_pair_tokens = max(
            len(
                reranker_model.tokenizer(
                    q,
                    t,
                    add_special_tokens=True,
                    truncation=False,
                )["input_ids"]
            )
            for q, t in pairs
        )
        if max_pair_tokens > reranker_limit:
            raise ValueError(
                "Un par query-fragmento excede el límite del reranker: "
                f"{max_pair_tokens} > {reranker_limit}."
            )

        scores = np.asarray(
            reranker_model.predict(
                pairs,
                batch_size=8,
                show_progress_bar=False,
            ),
            dtype=float,
        ).reshape(-1)
    else:
        scores = np.array(
            [offline_rerank_score(query, item["text"]) for item in candidates],
            dtype=float,
        )

    order = sorted(
        range(len(candidates)),
        key=lambda idx: (-float(scores[idx]), candidates[idx]["chunk_id"]),
    )

    return [
        {
            **candidates[int(idx)],
            "pre_rerank_rank": candidates[int(idx)]["rank"],
            "score": float(scores[int(idx)]),
            "rank": rank,
            "source": "reranker",
        }
        for rank, idx in enumerate(order, start=1)
    ]


#### **7. Construir los cuatro sistemas bajo el mismo protocolo**

Antes de medir se ejecuta un warm-up no registrado. La latencia reportada:

```text
incluye:
query-time retrieval
query embedding de dense
RRF
reranking

excluye:
carga de modelos
construcción de embeddings del corpus
construcción del índice
```

El notebook ejecuta las ramas BM25 y dense secuencialmente. Por tanto, la latencia end-to-end de `hybrid_RRF` representa esa ejecución secuencial; una implementación paralela tendría otro perfil.


In [ ]:
bm25_rankings = {}
dense_rankings = {}
hybrid_rankings = {}
reranked_rankings = {}
latency_rows = []

# Warm-up no registrado para reducir el efecto de la primera invocación.
if queries:
    warm = queries[0]
    warm_bm25 = bm25_ranking(warm["query"], depth=CANDIDATE_DEPTH)
    warm_dense = dense_ranking(warm["query_id"], depth=CANDIDATE_DEPTH)
    warm_hybrid = reciprocal_rank_fusion(
        [warm_bm25, warm_dense],
        depth=CANDIDATE_DEPTH,
        rank_constant=RRF_CONSTANT,
    )
    _ = rerank_candidates(warm["query"], warm_hybrid)

for row in queries:
    qid = row["query_id"]
    text = row["query"]

    t0 = time.perf_counter()
    bm25_list = bm25_ranking(text, depth=CANDIDATE_DEPTH)
    t1 = time.perf_counter()

    dense_list = dense_ranking(qid, depth=CANDIDATE_DEPTH)
    t2 = time.perf_counter()

    hybrid_list = reciprocal_rank_fusion(
        [bm25_list, dense_list],
        depth=CANDIDATE_DEPTH,
        rank_constant=RRF_CONSTANT,
    )
    t3 = time.perf_counter()

    reranked_list = rerank_candidates(text, hybrid_list)
    t4 = time.perf_counter()

    bm25_ms = (t1 - t0) * 1000.0
    dense_ms = (t2 - t1) * 1000.0
    rrf_ms = (t3 - t2) * 1000.0
    reranker_ms = (t4 - t3) * 1000.0

    latency_rows.append({
        "query_id": qid,
        "bm25_ms": bm25_ms,
        "dense_ms": dense_ms,
        "rrf_ms": rrf_ms,
        "reranker_ms": reranker_ms,
        "BM25_system_ms": bm25_ms,
        "dense_system_ms": dense_ms,
        "hybrid_RRF_system_ms": bm25_ms + dense_ms + rrf_ms,
        "hybrid_RRF_reranker_system_ms": (
            bm25_ms + dense_ms + rrf_ms + reranker_ms
        ),
    })

    bm25_rankings[qid] = bm25_list
    dense_rankings[qid] = dense_list
    hybrid_rankings[qid] = hybrid_list
    reranked_rankings[qid] = reranked_list

latency_per_query = pd.DataFrame(latency_rows)

def summarize_latency(series: pd.Series) -> dict[str, float]:
    values = series.to_numpy(dtype=float)
    return {
        "mean_ms": float(np.mean(values)),
        "p95_ms": float(np.percentile(values, 95)),
    }

latency_summary_rows = []
for system_name, column in [
    ("BM25", "BM25_system_ms"),
    ("dense", "dense_system_ms"),
    ("hybrid_RRF", "hybrid_RRF_system_ms"),
    ("hybrid_RRF_reranker", "hybrid_RRF_reranker_system_ms"),
]:
    latency_summary_rows.append({
        "system": system_name,
        **summarize_latency(latency_per_query[column]),
    })

latency_summary_df = pd.DataFrame(latency_summary_rows).set_index("system")

stage_latency_rows = []
for stage_name, column in [
    ("BM25", "bm25_ms"),
    ("dense", "dense_ms"),
    ("RRF", "rrf_ms"),
    ("reranker", "reranker_ms"),
]:
    stage_latency_rows.append({
        "stage": stage_name,
        **summarize_latency(latency_per_query[column]),
    })

stage_latency_summary_df = pd.DataFrame(stage_latency_rows).set_index("stage")

print("Rankings construidos:", len(queries), "consultas")
print("\nLatencia por etapa:")
display(stage_latency_summary_df)
print("\nLatencia end-to-end secuencial por sistema:")
latency_summary_df


#### **8. Recall@k sobre evidencia**

Para cada consulta:

$$
Recall@k=
\frac{\text{passages gold cubiertos por top-k fragmentos}}
{\text{passages gold totales}}.
$$

Un fragmento puede cubrir más de un `passage_id`.

El benchmark canónico exige al menos un `passage_id` relevante por consulta. `validate_dataset` falla temprano si encuentra qrels vacíos. `recall_at_k` conserva además una defensa local y devuelve `NaN` si recibe accidentalmente un gold vacío.

Después de las medias se calcula un **bootstrap pareado sobre queries** para la diferencia de `Recall@3`. El intervalo es evidencia descriptiva bajo este benchmark; no convierte el resultado en una afirmación universal.


In [ ]:
def recall_at_k(
    ranking: list[dict[str, Any]],
    gold_passages: list[str],
    k: int,
) -> float:
    retrieved_passages = {
        pid
        for item in ranking[:k]
        for pid in item["passage_ids"]
    }
    gold = set(gold_passages)
    if not gold:
        return float("nan")
    return len(gold & retrieved_passages) / len(gold)


def evaluate_system(
    rankings_by_query: dict[str, list[dict[str, Any]]],
    name: str,
) -> tuple[dict[str, float], pd.DataFrame]:
    rows = []
    for qid, gold in qrels.items():
        row = {"query_id": qid}
        for k in TOP_K_VALUES:
            row[f"recall@{k}"] = recall_at_k(
                rankings_by_query[qid],
                gold,
                k,
            )
        rows.append(row)

    per_query = pd.DataFrame(rows)
    summary = {"system": name}
    for k in TOP_K_VALUES:
        summary[f"Recall@{k}"] = float(per_query[f"recall@{k}"].mean())
    return summary, per_query


systems = {
    "BM25": bm25_rankings,
    "dense": dense_rankings,
    "hybrid_RRF": hybrid_rankings,
    "hybrid_RRF_reranker": reranked_rankings,
}

summaries = []
per_query_results = {}
for name, rankings in systems.items():
    summary, per_query = evaluate_system(rankings, name)
    summaries.append(summary)
    per_query_results[name] = per_query

summary_df = pd.DataFrame(summaries).set_index("system")
summary_df

#### **8.1. Incertidumbre: bootstrap pareado sobre consultas**

Se remuestrean las **mismas consultas** con reemplazo y se recalcula la diferencia de medias de `Recall@3`.

Comparaciones predefinidas:

```text
BM25 -> dense
dense -> hybrid_RRF
hybrid_RRF -> hybrid_RRF_reranker
```

El resultado reporta `system_b - system_a`, IC percentil 95% y si el intervalo excluye cero.


In [ ]:
def paired_bootstrap_mean_difference(
    values_a: np.ndarray,
    values_b: np.ndarray,
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    seed: int = SEED,
) -> dict[str, float | bool]:
    values_a = np.asarray(values_a, dtype=float)
    values_b = np.asarray(values_b, dtype=float)

    mask = np.isfinite(values_a) & np.isfinite(values_b)
    values_a = values_a[mask]
    values_b = values_b[mask]

    if len(values_a) == 0:
        raise ValueError("No hay pares válidos para bootstrap.")

    differences = values_b - values_a
    rng = np.random.default_rng(seed)
    sample_idx = rng.integers(
        0,
        len(differences),
        size=(n_resamples, len(differences)),
    )
    bootstrap_means = differences[sample_idx].mean(axis=1)

    ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
    observed = float(differences.mean())

    return {
        "mean_difference": observed,
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
        "ci95_excludes_zero": bool(ci_low > 0 or ci_high < 0),
        "n_queries": int(len(differences)),
    }


bootstrap_pairs = [
    ("BM25", "dense"),
    ("dense", "hybrid_RRF"),
    ("hybrid_RRF", "hybrid_RRF_reranker"),
]

bootstrap_rows = []
for pair_number, (system_a, system_b) in enumerate(bootstrap_pairs):
    a = per_query_results[system_a]["recall@3"].to_numpy(dtype=float)
    b = per_query_results[system_b]["recall@3"].to_numpy(dtype=float)

    result = paired_bootstrap_mean_difference(
        a,
        b,
        n_resamples=BOOTSTRAP_RESAMPLES,
        seed=SEED + pair_number,
    )
    bootstrap_rows.append({
        "system_a": system_a,
        "system_b": system_b,
        **result,
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df


#### **9. Análisis por consulta**

Una media puede ocultar ganancias y regresiones. Inspecciona al menos:

```text
1 consulta donde hybrid mejore
1 consulta donde no cambie
1 consulta donde reranking empeore, si existe
1 consulta ambiguous
```

La ausencia de un patrón también es un resultado.

In [ ]:
comparison = pd.DataFrame({
    "query_id": [q["query_id"] for q in queries],
    "difficulty": [q.get("difficulty", "unknown") for q in queries],
})

for name, df in per_query_results.items():
    comparison[name] = df.set_index("query_id").loc[
        comparison["query_id"], "recall@3"
    ].to_numpy()

comparison["hybrid_minus_dense"] = comparison["hybrid_RRF"] - comparison["dense"]
comparison["reranker_minus_hybrid"] = (
    comparison["hybrid_RRF_reranker"] - comparison["hybrid_RRF"]
)

comparison.sort_values(
    ["hybrid_minus_dense", "reranker_minus_hybrid"],
    ascending=[False, False],
)

In [ ]:
def inspect_query(query_id: str, top_k: int = 5) -> None:
    row = query_by_id[query_id]
    print("QUERY_ID:", query_id)
    print("Consulta:", row["query"])
    print("Gold passages:", qrels[query_id])
    print()

    for name, rankings in systems.items():
        print("SYSTEM:", name)
        for item in rankings[query_id][:top_k]:
            print(
                f"  rank={item['rank']} "
                f"chunk={item['chunk_id']} "
                f"score={item['score']:.6f} "
                f"passages={item['passage_ids']}"
            )
        print()


# Cambia el ID durante la clase para auditar un caso.
inspect_query(queries[0]["query_id"], top_k=3)

#### **10. RAG mínimo: ranking -> contexto -> LLM**

Se utiliza el pipeline final:

```text
BM25 + dense -> RRF -> Cross-Encoder -> top-3 -> contexto -> Qwen
```

Para que la integración no sea solo una demostración visual se añaden:

```text
1. baseline closed-book con el mismo LLM
2. RAG con el contexto recuperado
3. auditoría sintáctica de chunk_id citados
4. control negativo fuera del conocimiento del corpus
```

La auditoría de citas verifica **procedencia formal**: una cita es válida si el `chunk_id` apareció en el contexto. No decide si la oración está realmente sustentada por ese fragmento. Esa evaluación semántica corresponde a Semana 7.


In [ ]:
def build_context(
    ranking: list[dict[str, Any]],
    top_k: int = RAG_TOP_K,
) -> str:
    blocks = []
    for item in ranking[:top_k]:
        blocks.append(
            f"[{item['chunk_id']}] {item['title']}\n{item['text']}"
        )
    return "\n\n".join(blocks)


def offline_generate(question: str, context: str | None) -> str:
    if context:
        lines = [line.strip() for line in context.splitlines() if line.strip()]
        evidence = " ".join(lines[:4])
        return (
            "VALIDACION_OFFLINE: no es una respuesta del LLM canónico. "
            f"Pregunta={question} | Evidencia disponible={evidence[:500]}"
        )
    return (
        "VALIDACION_OFFLINE_CLOSED_BOOK: no es una respuesta del LLM canónico. "
        f"Pregunta={question}"
    )


if RUN_REAL_LLM:
    try:
        import torch
        import accelerate  # requerido por device_map="auto"
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as exc:
        raise ImportError(
            "La generación canónica requiere 'transformers', 'torch' y "
            "'accelerate'. Instala el entorno global del curso o exporta "
            "CC0F4_RUN_REAL_LLM=0 para validar offline."
        ) from exc

    tokenizer_kwargs = {}
    model_kwargs = {
        "dtype": "auto",
        "device_map": "auto",
    }
    if GENERATOR_MODEL_REVISION:
        tokenizer_kwargs["revision"] = GENERATOR_MODEL_REVISION
        model_kwargs["revision"] = GENERATOR_MODEL_REVISION

    generator_tokenizer = AutoTokenizer.from_pretrained(
        GENERATOR_MODEL_ID,
        **tokenizer_kwargs,
    )
    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_ID,
        **model_kwargs,
    )
else:
    generator_tokenizer = None
    generator_model = None


def generate_chat(system: str, user: str) -> str:
    if not RUN_REAL_LLM:
        raise RuntimeError("generate_chat solo se usa con el LLM real.")

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
    ).to(generator_model.device)

    with torch.no_grad():
        output = generator_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
        )

    generated = output[0, inputs["input_ids"].shape[1]:]
    return generator_tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


def generate_with_context(question: str, context: str) -> str:
    if not RUN_REAL_LLM:
        return offline_generate(question, context)

    system = (
        "Responde únicamente con información apoyada por el contexto. "
        "Cita los chunk_id entre corchetes. "
        "Si la evidencia es insuficiente, responde EVIDENCIA_INSUFICIENTE."
    )
    user = f"CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"
    return generate_chat(system, user)


def generate_closed_book(question: str) -> str:
    if not RUN_REAL_LLM:
        return offline_generate(question, None)

    system = (
        "Responde la pregunta sin recibir documentos externos. "
        "Si no conoces la respuesta, indícalo explícitamente."
    )
    return generate_chat(system, question)


CHUNK_CITATION_PATTERN = re.compile(r"\[([A-Za-z0-9_-]+-C\d+)\]")


def audit_chunk_citations(
    answer: str,
    allowed_chunk_ids: list[str],
) -> dict[str, Any]:
    cited = sorted(set(CHUNK_CITATION_PATTERN.findall(answer)))
    allowed = set(allowed_chunk_ids)
    invalid = sorted(cid for cid in cited if cid not in allowed)

    return {
        "cited_chunk_ids": cited,
        "invalid_chunk_ids": invalid,
        "all_citations_allowed": len(invalid) == 0,
        "evidence_insufficient": "EVIDENCIA_INSUFICIENTE" in answer,
    }


In [ ]:
rag_rows = []

available_demo_ids = [
    qid
    for qid in RAG_DEMO_QUERY_IDS
    if qid in query_by_id
]
if not available_demo_ids:
    available_demo_ids = [queries[0]["query_id"]]

for qid in available_demo_ids:
    question = query_by_id[qid]["query"]
    ranking = reranked_rankings[qid]
    context_ids = [item["chunk_id"] for item in ranking[:RAG_TOP_K]]
    context = build_context(ranking, top_k=RAG_TOP_K)

    t0 = time.perf_counter()
    closed_book_answer = generate_closed_book(question)
    t1 = time.perf_counter()
    rag_answer = generate_with_context(question, context)
    t2 = time.perf_counter()

    audit = audit_chunk_citations(rag_answer, context_ids)

    rag_rows.append({
        "query_id": qid,
        "question": question,
        "condition": "in_domain",
        "context_chunk_ids": context_ids,
        "gold_passages": qrels[qid],
        "closed_book_answer": closed_book_answer,
        "rag_answer": rag_answer,
        "closed_book_latency_ms": (t1 - t0) * 1000.0,
        "rag_generation_latency_ms": (t2 - t1) * 1000.0,
        **audit,
    })

# Control negativo: no se agrega a qrels ni a las métricas de retrieval.
negative_question = NEGATIVE_CONTROL["query"]
negative_bm25 = bm25_ranking(negative_question, depth=CANDIDATE_DEPTH)

# Dense usa el encoder sobre texto arbitrario sin incorporarlo al benchmark.
if RUN_REAL_RETRIEVAL:
    negative_qvec = dense_encoder.encode(
        [f"query: {negative_question}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype("float32")
    scores, indices = dense_index.search(
        negative_qvec,
        int(dense_index.ntotal),
    )
    negative_pairs = [
        (int(idx), float(score))
        for idx, score in zip(indices[0], scores[0])
        if int(idx) >= 0
    ]
    negative_pairs.sort(
        key=lambda pair: (-pair[1], chunks[pair[0]]["chunk_id"])
    )
    negative_dense = [
        {
            **chunks[idx],
            "score": score,
            "rank": rank,
            "source": "dense",
        }
        for rank, (idx, score) in enumerate(
            negative_pairs[:CANDIDATE_DEPTH],
            start=1,
        )
    ]
else:
    negative_qvec = dense_encoder.encode_queries([negative_question])
    negative_scores = document_vectors @ negative_qvec[0]
    negative_order = stable_desc_indices(
        negative_scores,
        chunks,
        CANDIDATE_DEPTH,
    )
    negative_dense = [
        {
            **chunks[idx],
            "score": float(negative_scores[idx]),
            "rank": rank,
            "source": "dense",
        }
        for rank, idx in enumerate(negative_order, start=1)
    ]

negative_hybrid = reciprocal_rank_fusion(
    [negative_bm25, negative_dense],
    depth=CANDIDATE_DEPTH,
    rank_constant=RRF_CONSTANT,
)
negative_reranked = rerank_candidates(
    negative_question,
    negative_hybrid,
)
negative_context_ids = [
    item["chunk_id"]
    for item in negative_reranked[:RAG_TOP_K]
]
negative_context = build_context(
    negative_reranked,
    top_k=RAG_TOP_K,
)

t0 = time.perf_counter()
negative_closed = generate_closed_book(negative_question)
t1 = time.perf_counter()
negative_rag = generate_with_context(
    negative_question,
    negative_context,
)
t2 = time.perf_counter()

negative_audit = audit_chunk_citations(
    negative_rag,
    negative_context_ids,
)

rag_rows.append({
    "query_id": NEGATIVE_CONTROL["query_id"],
    "question": negative_question,
    "condition": "negative_control",
    "context_chunk_ids": negative_context_ids,
    "gold_passages": [],
    "closed_book_answer": negative_closed,
    "rag_answer": negative_rag,
    "closed_book_latency_ms": (t1 - t0) * 1000.0,
    "rag_generation_latency_ms": (t2 - t1) * 1000.0,
    "expected_behavior": NEGATIVE_CONTROL["expected"],
    **negative_audit,
})

rag_df = pd.DataFrame(rag_rows)
rag_df


#### **10.1. Manifiesto reproducible de la corrida**

El cuaderno guarda una evidencia compacta en:

```text
Semana5/resultados/latest_run.json
```

Incluye:

```text
fecha UTC
seed
hashes SHA-256 del benchmark
IDs y revisiones resueltas de modelos cuando están disponibles
flags real/offline
configuración experimental
métricas agregadas
IC bootstrap
latencias
auditoría RAG
```

`resolved_commit` se toma del metadato del modelo cuando la biblioteca lo expone. No debe confundirse con un hash criptográfico de todos los pesos del modelo.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def package_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def resolved_model_commit(component: str) -> str | None:
    try:
        if component == "dense" and RUN_REAL_RETRIEVAL:
            config = dense_encoder[0].auto_model.config
            return getattr(config, "_commit_hash", None)
        if component == "reranker" and RUN_REAL_RERANKER:
            return getattr(reranker_model.model.config, "_commit_hash", None)
        if component == "generator" and RUN_REAL_LLM:
            return getattr(generator_model.config, "_commit_hash", None)
    except Exception:
        return None
    return None


def resolve_repo_root_for_results() -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if (root / "Semana4").is_dir() and (root / "README.md").is_file():
            return root
    return cwd


benchmark_hashes = {
    name: sha256_file(DATA_DIR / name)
    for name in [
        "corpus_semana4.jsonl",
        "queries_semana4.jsonl",
        "qrels_semana4.json",
    ]
}

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "benchmark_sha256": benchmark_hashes,
    "modes": {
        "real_retrieval": RUN_REAL_RETRIEVAL,
        "real_reranker": RUN_REAL_RERANKER,
        "real_llm": RUN_REAL_LLM,
    },
    "models": {
        "dense": {
            "id": DENSE_MODEL_ID,
            "requested_revision": DENSE_MODEL_REVISION,
            "resolved_commit": resolved_model_commit("dense"),
        },
        "reranker": {
            "id": RERANKER_MODEL_ID,
            "requested_revision": RERANKER_MODEL_REVISION,
            "resolved_commit": resolved_model_commit("reranker"),
        },
        "generator": {
            "id": GENERATOR_MODEL_ID,
            "requested_revision": GENERATOR_MODEL_REVISION,
            "resolved_commit": resolved_model_commit("generator"),
        },
    },
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scikit-learn": package_version("scikit-learn"),
        "rank-bm25": package_version("rank-bm25"),
        "sentence-transformers": package_version("sentence-transformers"),
        "faiss-cpu": package_version("faiss-cpu"),
        "transformers": package_version("transformers"),
        "accelerate": package_version("accelerate"),
    },
    "configuration": {
        "target_words": TARGET_WORDS,
        "overlap_passages": OVERLAP_PASSAGES,
        "candidate_depth": CANDIDATE_DEPTH,
        "top_k_values": TOP_K_VALUES,
        "rrf_constant": RRF_CONSTANT,
        "rag_top_k": RAG_TOP_K,
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    },
    "summary": (
        summary_df.reset_index()
        .to_dict(orient="records")
    ),
    "bootstrap_recall_at_3": bootstrap_df.to_dict(orient="records"),
    "latency_ms": {
        "stages": (
            stage_latency_summary_df.reset_index()
            .to_dict(orient="records")
        ),
        "systems_sequential": (
            latency_summary_df.reset_index()
            .to_dict(orient="records")
        ),
    },
    "rag_demo": rag_df.to_dict(orient="records"),
}

repo_root = resolve_repo_root_for_results()
results_dir = Path(
    os.environ.get(
        "CC0F4_RESULTS_DIR",
        str(repo_root / "Semana5" / "resultados"),
    )
).expanduser().resolve()
results_dir.mkdir(parents=True, exist_ok=True)

manifest_path = results_dir / "latest_run.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Manifiesto:", manifest_path)


#### **11. Preguntas de discusión**

1. ¿En qué consultas BM25 supera a dense y qué característica léxica podría explicarlo?
2. ¿RRF mejoró siempre respecto de ambos recuperadores individuales?
3. Si un passage relevante no aparece en los primeros `candidate_depth` resultados de ninguno de los retrievers, ¿puede recuperarlo el reranker?
4. ¿Por qué el Cross-Encoder se aplica después del retrieval?
5. Observa la tabla de latencia: ¿qué componente domina el costo y cómo cambiaría si `candidate_depth` creciera?
6. ¿Qué comparaciones tienen un IC bootstrap que excluye cero? ¿Qué puedes afirmar y qué no?
7. ¿El mismo LLM cambia su respuesta entre closed-book y RAG?
8. ¿El control negativo produce `EVIDENCIA_INSUFICIENTE` o fuerza una respuesta con evidencia irrelevante?
9. ¿Aparecen `chunk_id` citados que no estaban en el contexto?
10. ¿Por qué una cita formalmente válida todavía puede no sustentar semánticamente una afirmación?.

#### **Nota de atribución causal**

Las comparaciones no tienen todas la misma interpretación:

```text
BM25 vs dense
-> compara dos familias de recuperación

hybrid vs dense
-> añade simultáneamente la rama BM25 y una regla de fusión
-> no permite separar el efecto individual de cada elemento

hybrid vs hybrid + reranker
-> mantiene candidate pool y fusión
-> aísla de forma más limpia el efecto del reranker
```

Si se quisiera atribuir por separado el efecto de la señal BM25 y el de RRF, haría falta una ablación adicional.


#### **12. Conclusión limitada**

Completa después de ejecutar el modo canónico:

```text
Bajo este corpus, estas consultas, estos qrels, esta segmentación,
este encoder, este método de fusión y este reranker:

1. BM25 ...
2. dense ...
3. hybrid RRF ...
4. hybrid + reranker ...
5. diferencia con IC 95% ...
6. costo de latencia observado ...
7. caso de error observado ...
8. comportamiento del control negativo ...
9. conclusión que NO puede generalizarse ...
```

#### **Limitación del benchmark**

El corpus se hereda deliberadamente de Semana 4 para mantener continuidad experimental. No fue modificado después de observar los resultados de Semana 5.

Por ello, **no se afirma que sea un benchmark adversarial diseñado específicamente con distractores léxicos duros**. Si BM25 obtiene resultados altos, no debe concluirse que ese comportamiento se generaliza a corpus con entidades repetidas, terminología ambigua o documentos casi duplicados.

Agregar distractores después de ver estos resultados mezclaría diseño del benchmark con observación post hoc. Una futura comparación con distractores debe declararse como una condición experimental nueva.

#### **Criterio de cierre**

Debes poder explicar sin depender del notebook:

```text
BM25 -------+
             +-> RRF -> Cross-Encoder -> top-k -> contexto -> LLM
E5 + FAISS --+
```

y distinguir:

```text
retrieval != reranking != generation

Recall@k != calidad final de RAG

cita válida != groundedness demostrado

diferencia de medias != evidencia estadística suficiente

latencia de notebook != benchmark de producción
```

La siguiente semana introduce herramientas con contratos, validación y manejo de errores.
